<a href="https://colab.research.google.com/github/yogahelmi/Grokking-Deep-Learning/blob/main/05.generalizing-gradient-descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 05: Generalizing Gradient Descent - Learning Multiple Weights at a Time
**Enrichment for Deep Learning Classes - Individual Task 2: Code Reproduction & Theoretical Deep-Dive**  
**Author:** Yoga Helmi Ilham Saputra  
**GitHub:** [yogahelmi/Grokking-Deep-Learning](https://github.com/yogahelmi/Grokking-Deep-Learning)  
**Primary References:**
- *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)
- *Practical Linear Algebra for Data Science* by Mike X Cohen (O'Reilly Media)

---

## 1. Executive Summary & Overview
Chapter 5 expands gradient descent from adjusting a single scalar weight to optimizing multidimensional weight vectors and weight matrices simultaneously.

We explore three progressive configurations:
1. **Multiple Inputs $\rightarrow$ Single Output:** Learning a weight vector $\mathbf{w} \in \mathbb{R}^n$.
2. **The Credit Assignment Dilemma (Freezing Weights):** How freezing one parameter forces the remaining parameters to compensate.
3. **Single Input $\rightarrow$ Multiple Outputs:** Distributing one signal to multiple tasks with individual error deltas.
4. **Multiple Inputs $\rightarrow$ Multiple Outputs (Full Matrix Update):** Optimizing a weight matrix $\mathbf{W} \in \mathbb{R}^{m \times n}$ via **outer product** updates.

### Key Chapter Takeaways:
- **Gradient Vector:** In multiple dimensions, the derivative becomes the gradient vector $\nabla_\mathbf{w} E = \delta \cdot \mathbf{x}$.
- **Matrix Update via Outer Product:** In a network with multiple inputs and multiple outputs, the gradient of the error with respect to the weight matrix is the outer product of the output error vector $\mathbf{\delta}$ and the input vector $\mathbf{x}$:
  $$\Delta \mathbf{W} = \mathbf{\delta} \otimes \mathbf{x} = \mathbf{\delta} \mathbf{x}^T$$


---
## 2. Theoretical Deep-Dive: Multidimensional Gradient Mechanics

### 2.1 Multiple Inputs, Single Output
Consider input vector $\mathbf{x} = [x_1, x_2, \dots, x_n]^T$ and weight vector $\mathbf{w} = [w_1, w_2, \dots, w_n]^T$:
$$\hat{y} = \mathbf{w}^T \mathbf{x} = \sum_{i=1}^n w_i x_i$$
The squared error is $E = (\hat{y} - y)^2$.
Using the multivariate chain rule, the partial derivative with respect to each individual weight $w_i$ is:
$$\frac{\partial E}{\partial w_i} = \frac{\partial E}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial w_i} = 2(\hat{y} - y) \cdot x_i$$
We collect all partial derivatives into the **gradient vector**:
$$\nabla_\mathbf{w} E = \begin{bmatrix} \frac{\partial E}{\partial w_1} \\ \frac{\partial E}{\partial w_2} \\ \vdots \\ \frac{\partial E}{\partial w_n} \end{bmatrix} = 2(\hat{y} - y) \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{bmatrix} = 2 \delta \mathbf{x}$$
The vector update rule is:
$$\mathbf{w} \leftarrow \mathbf{w} - \alpha \nabla_\mathbf{w} E$$

### 2.2 The Credit Assignment Problem & Freezing Weights
When multiple inputs contribute to a single prediction, gradient descent shares the credit/blame across all active features.
- If we **freeze** one weight (e.g., $w_1$ held constant), the remaining weights ($w_2, \dots, w_n$) adapt to absorb the residual error.
- This demonstrates that neural network parameters do not find unique independent solutions; they find collaborative points on a multidimensional manifold that satisfy the loss criterion.

### 2.3 Multiple Inputs, Multiple Outputs (The Outer Product Matrix Update)
Now consider the general case: $n$ inputs and $m$ outputs governed by weight matrix $\mathbf{W} \in \mathbb{R}^{m \times n}$:
$$\mathbf{\hat{y}} = \mathbf{W} \mathbf{x}$$
Each output unit $j$ has its own target $y_j$ and error delta:
$$\delta_j = \hat{y}_j - y_j$$
Collecting all output deltas into vector $\mathbf{\delta} \in \mathbb{R}^m$:
$$\mathbf{\delta} = \begin{bmatrix} \delta_1 \\ \delta_2 \\ \vdots \\ \delta_m \end{bmatrix}$$
The partial derivative with respect to weight entry $W_{ji}$ (connecting input $i$ to output $j$) is:
$$\frac{\partial E}{\partial W_{ji}} = \delta_j \cdot x_i$$
In matrix notation, this is the **Outer Product** ($\otimes$):
$$\nabla_\mathbf{W} E = \mathbf{\delta} \mathbf{x}^T = \begin{bmatrix} \delta_1 \\ \delta_2 \\ \vdots \\ \delta_m \end{bmatrix} \begin{bmatrix} x_1 & x_2 & \cdots & x_n \end{bmatrix} = \begin{bmatrix} 
\delta_1 x_1 & \delta_1 x_2 & \cdots & \delta_1 x_n \\
\delta_2 x_1 & \delta_2 x_2 & \cdots & \delta_2 x_n \\
\vdots & \vdots & \ddots & \vdots \\
\delta_m x_1 & \delta_m x_2 & \cdots & \delta_m x_n
\end{bmatrix}$$

#### Practical Linear Algebra Insight:
The outer product $\mathbf{\delta} \mathbf{x}^T$ is a **Rank-1 Matrix**! Every single training instance generates a rank-1 update matrix that is added to $\mathbf{W}$. Over many iterations and varied inputs, the sum of these rank-1 updates forms a full-rank transformation capturing complex multidimensional correlations.


---
## 3. Code Reproduction: Multi-Weight Gradient Descent

### 3.1 Gradient Descent with Multiple Inputs
We predict win/loss outcome from toes, win%, and fans:


In [1]:
import numpy as np

def w_sum(a, b):
    assert(len(a) == len(b))
    output = 0
    for i in range(len(a)):
        output += (a[i] * b[i])
    return output

weights = [0.1, 0.2, -0.1]

toes = [8.5, 9.5, 9.9, 9.0]
wlrec = [0.65, 0.8, 0.8, 0.9]
nfans = [1.2, 1.3, 0.5, 1.0]

win_or_lose_binary = [1, 1, 0, 1]

true_target = win_or_lose_binary[0]
input_features = [toes[0], wlrec[0], nfans[0]]

def neural_network(input_vec, weights_vec):
    return w_sum(input_vec, weights_vec)

def ele_mul_scalar(scalar, vector):
    output = [0] * len(vector)
    for i in range(len(vector)):
        output[i] = scalar * vector[i]
    return output

alpha = 0.01

print("--- Starting Multiple Inputs Gradient Descent ---")
for iteration in range(4):
    pred = neural_network(input_features, weights)
    error = (pred - true_target) ** 2
    delta = pred - true_target
    weight_deltas = ele_mul_scalar(delta, input_features)
    
    print(f"Iteration {iteration} | Error: {error:.4f} | Pred: {pred:.4f}")
    print(f"  Weights: {[round(w, 4) for w in weights]}")
    print(f"  Weight Deltas: {[round(wd, 4) for wd in weight_deltas]}")
    
    for i in range(len(weights)):
        weights[i] -= alpha * weight_deltas[i]

--- Starting Multiple Inputs Gradient Descent ---
Iteration 0 | Error: 0.0324 | Pred: 0.8200
  Weights: [0.1, 0.2, -0.1]
  Weight Deltas: [-1.53, -0.117, -0.216]
Iteration 1 | Error: 0.0030 | Pred: 0.9455
  Weights: [0.1153, 0.2012, -0.0978]
  Weight Deltas: [-0.4633, -0.0354, -0.0654]
Iteration 2 | Error: 0.0003 | Pred: 0.9835
  Weights: [0.1199, 0.2015, -0.0972]
  Weight Deltas: [-0.1402, -0.0107, -0.0198]
Iteration 3 | Error: 0.0000 | Pred: 0.9950
  Weights: [0.1213, 0.2016, -0.097]
  Weight Deltas: [-0.0424, -0.0032, -0.006]

### 3.2 Freezing One Weight: Observing Credit Compensation
What happens if we freeze `weights[0]` (hold toes weight constant at 0.1)?
The other two weights (`wlrec` and `nfans`) will adjust further to compensate for the frozen parameter:


In [1]:
# Reset weights and freeze weight 0
weights_frozen = [0.1, 0.2, -0.1]

print("--- Freezing Weight 0 (Toes) ---")
for iteration in range(4):
    pred = neural_network(input_features, weights_frozen)
    error = (pred - true_target) ** 2
    delta = pred - true_target
    weight_deltas = ele_mul_scalar(delta, input_features)
    
    # Freeze index 0
    weight_deltas[0] = 0.0
    
    print(f"Iteration {iteration} | Error: {error:.4f} | Pred: {pred:.4f}")
    print(f"  Weights: {[round(w, 4) for w in weights_frozen]}")
    
    for i in range(len(weights_frozen)):
        weights_frozen[i] -= alpha * weight_deltas[i]

--- Freezing Weight 0 (Toes) ---
Iteration 0 | Error: 0.0324 | Pred: 0.8200
  Weights: [0.1, 0.2, -0.1]
Iteration 1 | Error: 0.0270 | Pred: 0.8356
  Weights: [0.1, 0.2012, -0.0978]
Iteration 2 | Error: 0.0225 | Pred: 0.8499
  Weights: [0.1, 0.2022, -0.0959]
Iteration 3 | Error: 0.0188 | Pred: 0.8629
  Weights: [0.1, 0.2031, -0.0941]

### 3.3 Gradient Descent with Multiple Outputs
Single input (`wlrec = 0.65`) predicting three outputs (`[hurt, win, sad]`):


In [1]:
weights_1_to_3 = [0.3, 0.2, 0.9]
wlrec_val = 0.65
true_targets_3 = [0.1, 1.0, 0.1]  # [hurt, win, sad]
alpha = 0.1

def neural_net_single_in(inp, wgts):
    return [inp * w for w in wgts]

print("--- Starting Multiple Outputs Gradient Descent ---")
for iteration in range(4):
    pred = neural_net_single_in(wlrec_val, weights_1_to_3)
    error = [(p - t) ** 2 for p, t in zip(pred, true_targets_3)]
    deltas = [p - t for p, t in zip(pred, true_targets_3)]
    weight_deltas = [d * wlrec_val for d in deltas]
    
    print(f"Iteration {iteration} | Total Error: {sum(error):.4f}")
    print(f"  Predictions: {[round(p, 4) for p in pred]}")
    print(f"  Weights: {[round(w, 4) for w in weights_1_to_3]}")
    
    for i in range(len(weights_1_to_3)):
        weights_1_to_3[i] -= alpha * weight_deltas[i]

--- Starting Multiple Outputs Gradient Descent ---
Iteration 0 | Total Error: 0.9997
  Predictions: [0.195, 0.13, 0.585]
  Weights: [0.3, 0.2, 0.9]
Iteration 1 | Total Error: 0.9169
  Predictions: [0.1888, 0.1866, 0.5535]
  Weights: [0.2938, 0.2566, 0.8685]
Iteration 2 | Total Error: 0.8409
  Predictions: [0.1832, 0.2394, 0.524]
  Weights: [0.288, 0.3093, 0.839]
Iteration 3 | Total Error: 0.7712
  Predictions: [0.1779, 0.2889, 0.4965]
  Weights: [0.2826, 0.3587, 0.8115]

### 3.4 Multiple Inputs & Multiple Outputs: Outer Product Updates
Full matrix gradient descent connecting 3 inputs to 3 outputs:


In [1]:
# Weights Matrix (3 outputs x 3 inputs)
weights_matrix = np.array([
    [0.1, 0.1, -0.3],  # hurt?
    [0.1, 0.2, 0.0],   # win?
    [0.0, 1.3, 0.1]    # sad?
])

inputs_vec = np.array([8.5, 0.65, 1.2])
targets_vec = np.array([0.1, 1.0, 0.1])
alpha = 0.01

print("--- Starting Multiple-Input Multiple-Output Gradient Descent ---")
for iteration in range(4):
    # Step 1: Forward Propagation (y_hat = W * x)
    predictions = weights_matrix.dot(inputs_vec)
    
    # Step 2: Compare (Error & Deltas)
    errors = (predictions - targets_vec) ** 2
    deltas = predictions - targets_vec
    
    # Step 3: Learn via Outer Product: deltas (3,) outer inputs (3,) -> (3, 3)
    gradient_matrix = np.outer(deltas, inputs_vec)
    
    print(f"Iteration {iteration} | Total Error: {np.sum(errors):.4f}")
    print(f"  Predictions: {np.round(predictions, 4)}")
    
    # Update weight matrix
    weights_matrix -= alpha * gradient_matrix

print("\nFinal Optimized Weight Matrix:")
print(np.round(weights_matrix, 4))

--- Starting Multiple-Input Multiple-Output Gradient Descent ---
Iteration 0 | Total Error: 0.9555
  Predictions: [0.555 0.98  0.965]
Iteration 1 | Total Error: 0.0053
  Predictions: [0.1341 0.9985 0.165]
Iteration 2 | Total Error: 0.0000
  Predictions: [0.1026 0.9999 0.1049]
Iteration 3 | Total Error: 0.0000
  Predictions: [0.1002 1.     0.1004]

Final Optimized Weight Matrix:
[[ 0.0655  0.0974 -0.3049]
 [ 0.1003  0.2     0.    ]
 [-0.0657  1.295   0.0907]]

---
## 4. Practical Linear Algebra: Rank-1 Matrix Updates
Let us inspect the outer product matrix $\mathbf{\delta} \mathbf{x}^T$ directly and verify its algebraic rank:


In [1]:
sample_delta = np.array([0.455, -0.02, 0.865])
sample_input = np.array([8.5, 0.65, 1.2])

outer_prod_mat = np.outer(sample_delta, sample_input)
mat_rank = np.linalg.matrix_rank(outer_prod_mat)

print("Outer Product Matrix (delta (x) input):")
print(np.round(outer_prod_mat, 4))
print(f"Algebraic Matrix Rank: {mat_rank}")
assert mat_rank == 1, "The outer product of two non-zero vectors is strictly rank 1!\"

Outer Product Matrix (delta (x) input):
[[ 3.8675  0.2958  0.546 ]
 [-0.17   -0.013  -0.024 ]
 [ 7.3525  0.5622  1.038 ]]
Algebraic Matrix Rank: 1

---
## 5. Chapter Review & Synthesis Questions

### Conceptual Self-Check:
1. **Q: Why is the gradient of a weight matrix computed using the vector outer product $\mathbf{\delta} \mathbf{x}^T$?**  
   *A:* Each individual weight $W_{ji}$ connects input node $x_i$ to output node $y_j$. By the chain rule, $\frac{\partial E}{\partial W_{ji}} = \frac{\partial E}{\partial \hat{y}_j} \frac{\partial \hat{y}_j}{\partial W_{ji}} = \delta_j x_i$. Gathering these partial derivatives across all $i$ and $j$ forms the outer product matrix $\mathbf{\delta} \mathbf{x}^T$.

2. **Q: What happens when one weight is frozen during multi-variable gradient descent?**  
   *A:* The unconstrained weights continue gradient descent, adapting to fit the training data while taking the frozen weight's contribution as a constant baseline. This illustrates parameter co-adaptation.

3. **Q: Why does a single sample update produce a rank-1 weight adjustment, yet the final weight matrix is full rank?**  
   *A:* Each training sample produces a rank-1 outer product $\mathbf{\delta}^{(k)} (\mathbf{x}^{(k)})^T$. Summing distinct rank-1 update matrices over diverse training examples spans a multidimensional subspace, accumulating rank up to $\min(m, n)$.
